In [1]:
!pip install torch_scatter torcheeg torch_geometric -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 57.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset ,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
import torch_geometric.loader as geom_loader 
import copy
import scipy.signal as signal
import random
import numpy as np
import os

In [3]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Set a fixed random seed for reproducibility across different libraries.
def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [5]:
def map_emotions(y):
    if y == 0: 
        return 0  # Neutral remains 0
    elif y == 1 or y == 2: 
        return 1  # Sad (1) and Fear (2) become Negative (1)
    elif y == 3: 
        return 2  # Happy (3) becomes Positive (2)
    return 0

In [6]:
dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'], 
    num_worker=0,
    offline_transform=transforms.Compose([
        transforms.To2d(), 
        transforms.Lambda(lambda x: torch.tensor(x).float())]),
    label_transform=transforms.Compose([
        transforms.Select('emotion'),
        transforms.Lambda(map_emotions)
    ])
   
)

[2026-03-01 15:59:46] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
[2026-03-01 15:59:46] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 1it [00:00,  6.00it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 6it [00:00, 23.56it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 9it [00:01,  4.49it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 11it [00:03,  2.38it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 12it [00:04,  2.18it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat

In [7]:
# This DataFrame contains columns like: subject_id, trial_id, emotion, etc.
meta_info = dataset.info 
all_subjects = sorted(meta_info['subject_id'].unique())
print(f"Subjects Found: {all_subjects}")

Subjects Found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [8]:
# Parameters
EPOCHS = 100
BATCH_SIZE = 16
LR = 0.005
# PATIENCE_LR = 3
# REDUCE_FACTOR = 0.5
PATIENCE_ES = 25
WEIGHT_DECAY = 0.0001
HIDE_CHANNELS = 128
LABEL_SMOOTHING = 0.1
NUM_LAYERS = 3

In [9]:
# Subject-Dependent Loop
from sklearn.model_selection import train_test_split

# Create a directory to save the best models
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')

subject_results = {}

print("Start Subject-Dependent Evaluation...")

for subject_id in all_subjects:
    print(f"\n{'='*30} Processing Subject {subject_id} {'='*30}")
    
    # Create a unique ID for each recording (session + trial)
    # Since SEED IV repeats trial IDs (1-24) across sessions, 
    # merging them ensures we recognize all 72 trials per subject
    subject_df = meta_info[meta_info['subject_id'] == subject_id].copy()
    subject_df['unique_run_id'] = subject_df['session_id'].astype(str) + "_" + subject_df['trial_id'].astype(str)
    
    # Extract labels per unique run to ensure balanced split
    # We get a single label for each unique run (session_trial)
    run_info = subject_df[['unique_run_id', 'emotion']].drop_duplicates()
    all_runs = run_info['unique_run_id'].values
    all_labels = run_info['emotion'].values
    
    # Stratified Split at the RUN level 
    # This keeps 80/20 ratio and ensures each class is represented in the test set
    train_runs, test_runs = train_test_split(
        all_runs, 
        test_size=0.20, 
        random_state=42, 
        stratify=all_labels
    )
    
    print(f"Total Unique Videos (Across 3 Sessions): {len(all_runs)}")
    print(f"Training on: {len(train_runs)} | Testing on: {len(test_runs)}")
    
    # Extract indices (Zero Leakage Guaranteed)
    train_indices = subject_df[subject_df['unique_run_id'].isin(train_runs)].index.tolist()
    test_indices = subject_df[subject_df['unique_run_id'].isin(test_runs)].index.tolist()
    
    # Dataset Subsets and Loaders
    train_set = Subset(dataset, train_indices)
    test_set = Subset(dataset, test_indices)
    
    # Weighted Sampler for further balancing during training 
    y_train = meta_info.iloc[train_indices]['emotion'].values
    mapped_labels = [map_emotions(y) for y in y_train]
    class_counts = np.bincount( mapped_labels)
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[y] for y in  mapped_labels]
    
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- 3. Initialize Model ---
    model = DGCNN(
        in_channels=5, 
        num_electrodes=62, 
        num_layers=NUM_LAYERS, 
        hid_channels=HIDE_CHANNELS, 
        num_classes=3,
    ).to(device)
    
    criterion = nn.CrossEntropyLoss( label_smoothing = LABEL_SMOOTHING)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=REDUCE_FACTOR, patience=PATIENCE_LR)

    # optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
   
    # Training Loop 
    best_val_acc = 0.0
    counter = 0 # For early stopping
    
    for epoch in range(EPOCHS):
        model.train()
        correct = 0
        total = 0
        train_loss = 0
        
        for X, y in train_loader:
            X = X.to(device).float()
            y = y.to(device).long()
            
            if len(X.shape) == 4: X = X.squeeze(1) 
          
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
       
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * correct / total
            
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device).float()
                y = y.to(device).long()
               
                if len(X.shape) == 4: X = X.squeeze(1)
                
                outputs = model(X)
                val_loss += criterion(outputs, y).item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += y.size(0)
                val_correct += (predicted == y).sum().item()
        
        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(test_loader)
        #scheduler.step(avg_val_loss)
        scheduler.step()

        print(f"Epoch {epoch+1:02d} | "
              f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}%")
        
        # Early Stopping Check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0

            save_path = f"saved_models/subject_{subject_id}_best.pth"
            torch.save(model.state_dict(), save_path)
            
        else:
            counter += 1
            if counter >= PATIENCE_ES:
                print(f"Early stopping at epoch {epoch}")
                break
                
    print(f"Subject {subject_id} Best Acc: {best_val_acc:.2f}%")
    subject_results[subject_id] = best_val_acc

# ================= FINAL REPORT =================
print("\n" + "="*40)
print("FINAL RESULTS REPORT ")
print("="*40)

all_accuracies = list(subject_results.values())

for sub_id in sorted(subject_results.keys()):
    print(f"Subject {sub_id}: {subject_results[sub_id]:.2f}%")

print(f"\nAverage Accuracy: {np.mean(all_accuracies):.2f}%")

if len(subject_results) > 0:
    best_sub_id = max(subject_results, key=subject_results.get)
    print(f"HIGHEST ACCURACY: Subject {best_sub_id} ({subject_results[best_sub_id]:.2f}%)")
print("="*40)

Start Subject-Dependent Evaluation...

============================== Processing Subject 1 ==============================
Total Unique Videos (Across 3 Sessions): 72
Training on: 57 | Testing on: 15
Epoch 01 | Train Loss: 0.6501 Acc: 84.61% | Val Loss: 1.7153 Acc: 49.66%
Epoch 02 | Train Loss: 0.3501 Acc: 98.49% | Val Loss: 0.7249 Acc: 75.95%
Epoch 03 | Train Loss: 0.3030 Acc: 100.00% | Val Loss: 0.8021 Acc: 77.84%
Epoch 04 | Train Loss: 0.3032 Acc: 99.95% | Val Loss: 0.7220 Acc: 81.10%
Epoch 05 | Train Loss: 0.2969 Acc: 100.00% | Val Loss: 0.8260 Acc: 71.13%
Epoch 06 | Train Loss: 0.2965 Acc: 100.00% | Val Loss: 0.7587 Acc: 79.38%
Epoch 07 | Train Loss: 0.2971 Acc: 100.00% | Val Loss: 0.7512 Acc: 81.10%
Epoch 08 | Train Loss: 0.2953 Acc: 100.00% | Val Loss: 0.7182 Acc: 77.84%
Epoch 09 | Train Loss: 0.2953 Acc: 100.00% | Val Loss: 0.6957 Acc: 75.95%
Epoch 10 | Train Loss: 0.2955 Acc: 100.00% | Val Loss: 0.7822 Acc: 75.60%
Epoch 11 | Train Loss: 0.2960 Acc: 100.00% | Val Loss: 0.7494 Ac